# 01 — Simulated Single-Sérsic Single-Band Fit

This notebook generates one controlled Single-Sérsic observation, runs the complete production single-band workflow, and reports posterior summaries together with the injected parameters.

All figures are produced by the BayMorph plotting system.

In [ ]:
from pathlib import Path
import math
import os

from baymorph import BayMorphRunner
from baymorph.pipeline import run_simulation_from_config

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

CONFIG = Path("configs/simulated_galaxy.yaml")

## Generate the controlled observation

The YAML configuration defines the intrinsic galaxy, PSF, background, noise model, priors, and inference-backend configuration.

The public configuration uses pocoMC by default. Sampler-specific options retain the native names and semantics of the selected upstream library.


In [ ]:
run_simulation_from_config(CONFIG)

## Inference backend

BayMorph defines the scientific inference problem independently of the numerical backend.

The default configuration uses **pocoMC**. Switching to Dynesty requires changing only the `fit.sampler` block in the YAML configuration; the morphology model, priors, likelihood, and scientific interpretation remain unchanged.

For Dynesty, `mode: static` selects `dynesty.NestedSampler`, while `mode: dynamic` selects `dynesty.DynamicNestedSampler`.

Options under `init` and `run` follow the native upstream API. Consult the documentation for the installed pocoMC or Dynesty version for the complete set of numerical options.


## Run the complete BayMorph workflow

This is the recommended user-facing path. It performs image preparation, PSF handling, Bayesian fitting, persistence, and the configured diagnostics.

In [ ]:
runner = BayMorphRunner(CONFIG)
fit = runner.run(diagnostics=True)

## Posterior summary

`fit` contains the scientific products of the morphology fit. Its `fit.posterior` attribute is the canonical backend-neutral `InferenceResult`.

The injected position angle is converted from degrees to radians to match the fitted parameter convention. The maximum-likelihood sample is reported only as a secondary diagnostic.


In [ ]:
truth = dict(runner.config["simulation"]["truth"])
truth["theta"] = math.radians(truth.pop("theta_deg"))

posterior_summary = {
    name: fit.posterior.summary(name)
    for name in fit.posterior.parameter_names
}

{
    "truth_parameters_in_fit_units": truth,
    "posterior_summary": posterior_summary,
    "maximum_likelihood_parameters": fit.best_parameters,
    "fit_statistics": fit.statistics,
    "output_directory": str(runner.run_dir),
}

## Equivalent step-by-step workflow

The all-in-one command above is recommended. The same analysis can be executed explicitly when intermediate products must be inspected:

```python
runner = BayMorphRunner(CONFIG)
runner.prepare_image()
# For psf.type: empirical only:
# runner.select_psf_candidates()
# runner.build_psf()
fit = runner.fit()
runner.diagnostics()
```

For the Gaussian PSF used here, `runner.run(diagnostics=True)` automatically skips empirical-PSF candidate selection and construction.